# পাঠ ১১ - এজেন্ট-থেকে-এজেন্ট (এ২এ) প্রোটোকল


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## বিশেষায়িত ট্রাভেল এজেন্ট তৈরি করা


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ওয়ার্কফ্লোর মাধ্যমে মাল্টি-এজেন্ট সহযোগিতা

আমরা তিনটি এজেন্টকে একটি ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করি যা A2A মেসেজ পাসিং-এর অনুকরণ করে:

1. **CurrencyExchangeAgent** ব্যবহারকারীর অনুরোধ গ্রহণ করে এবং মুদ্রা পরামর্শ প্রদান করে।
2. **ActivityPlannerAgent** সমৃদ্ধ প্রসঙ্গ গ্রহণ করে এবং কার্যকলাপের সুপারিশ যোগ করে।
3. **TravelManagerAgent** উভয় ইনপুটকে সমন্বিত করে একটি চূড়ান্ত ভ্রমণ সংক্ষিপ্তসার প্রস্তুত করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## প্রোডাকশন-এ A2A বোঝা

প্রোডাকশন পরিবেশে A2A প্রোটোকল শক্তিশালী ক্রস-সার্ভিস দৃশ্যপট গুলো উন্মোচন করে:

| সক্ষমতা | বর্ণনা |
|---|---|
| **ক্রস-ফ্রেমওয়ার্ক ইন্টারঅপ** | এক ফ্রেমওয়ার্ক দিয়ে নির্মিত একটি এজেন্ট যেকোনো অন্য A2A-সঙ্গত ফ্রেমওয়ার্ক দিয়ে নির্মিত এজেন্টকে কাজ বাৎসরিক করে দিতে পারে, যা সত্যিকার অর্থে ক্রস-সংগঠন আন্তঃপরিচালন সক্ষম করে। |
| **সার্ভিস সীমানা** | এজেন্টগুলি আলাদা মাইক্রোসার্ভিস, ক্লাউড অঞ্চল বা এমনকি আলাদা আলাদা প্রতিষ্ঠানেও থাকতে পারে এবং তবুও নির্বিঘ্নে একত্রে কাজ করতে পারে। |
| **ডায়নামিক ডিসকভারী** | একটি অর্কেস্ট্রেটর রানটাইম-এ একটি এজেন্ট কার্ড রেজিস্ট্রিতে অনুসন্ধান করতে পারে যাতে নির্দিষ্ট উপ-কার্যের জন্য সেরা-মামলা বিশেষজ্ঞ খুঁজে পাওয়া যায়। |
| **স্ট্রিমিং ও পুশ নোটিফিকেশন** | A2A রিয়েল-টাইম অগ্রগতির আপডেট এবং দীর্ঘস্থায়ী কাজের জন্য পুশ নোটিফিকেশন সাপোর্ট করে সার্ভার-সেন্ট ইভেন্টস (SSE) এর মাধ্যমে। |

উপরে আমরা যে ওয়ার্কফ্লো তৈরি করেছি তা এই প্যাটার্নের একটি সরলীকৃত, প্রসেসের মধ্যে সংস্করণ। বাস্তব
ডিপ্লয়মেন্ট-এ প্রতিটি এজেন্ট একটি HTTP এন্ডপয়েন্ট উন্মুক্ত করবে, একটি এজেন্ট কার্ড প্রকাশ করবে এবং
A2A JSON-RPC প্রোটোকলের মাধ্যমে যোগাযোগ করবে।


## সারাংশ

এই পাঠে আপনি শিখেছেন:

1. **A2A প্রোটোকল কী** — এজেন্ট থেকে এজেন্ট আবিষ্কার, মেসেজিং, এবং কাজ ব্যবস্থাপনার জন্য একটি খোলা মানদণ্ড।
   এবং টাস্ক ম্যানেজমেন্ট।
2. **কিভাবে বিশেষায়িত এজেন্ট তৈরি করবেন** — একটি কারেন্সি এক্সচেঞ্জ এজেন্ট, একটি অ্যাক্টিভিটি প্লানার এজেন্ট, এবং একটি ট্র্যাভেল ম্যানেজার অর্কেস্ট্রেটর।

3. **কিভাবে এজেন্টদের ওয়ার্কফ্লোতে সংযুক্ত করবেন** — `WorkflowBuilder` ব্যবহার করে এজেন্টদের মধ্যে ধারাবাহিক মেসেজ পাঠানোর মডেল তৈরি করা।

4. **A2A কিভাবে প্রোডাকশনে কাজ করে** — ডায়নামিক আবিষ্কার এবং স্ট্রিমিং আপডেটের মাধ্যমে ফ্রেমওয়ার্ক জুড়ে, সার্ভিস জুড়ে সহযোগিতা সক্ষম করা।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
